# FastTD3 en Colab gratis, solo simulación — humanoide Booster T1

Reimplementación de **FastTD3** (Seo et al., 2025, [arXiv 2505.22642](https://arxiv.org/abs/2505.22642)) lo más parecida posible al código oficial
([younggyoseo/FastTD3](https://github.com/younggyoseo/FastTD3), licencia MIT), pensada para una GPU T4 gratuita.
Entrena **y** hace inferencia, todo en simulación (sin pasar al robot real).

**Qué hace**
1. Instala lo necesario (MuJoCo Playground 0.2.0, JAX, PyTorch).
2. Mide la velocidad de tu GPU (celda de *benchmark*) y te sugiere cuántas iteraciones caben en tu tiempo.
3. Entrena al humanoide **Booster T1** (`T1JoystickFlatTerrain`). La validación en una tarea sencilla (CheetahRun) está en el otro cuaderno.
4. **Cada 1/4 del total de iteraciones muestra aquí mismo un video del robot** con la política actual, para ver cómo aprende.
5. Guarda checkpoints, evalúa y genera un video final de inferencia.

**Qué se conserva del código oficial:** actor y crítico (MLP), crítico distribucional C51, doble Q con mínimo (CDQ),
ruido de exploración distinto por entorno, ruido de política objetivo recortado, actualización retardada del actor,
retornos de n pasos, normalización de observaciones, buffer de repetición en GPU, AdamW con weight decay 0,1,
actualización suave del crítico objetivo, observaciones privilegiadas para el crítico (actor-crítico asimétrico).

**Qué cambia (y por qué):** menos entornos, lotes más pequeños y buffer más corto, para caber en ~15 GB de una T4;
precisión mixta fp16 con `GradScaler` (la T4 no tiene bf16 nativo); `torch.compile` desactivado por defecto; sin wandb;
sin SimNorm ni SimbaV2 (son opciones del repositorio que están apagadas por defecto); diccionarios en lugar de `TensorDict`.
Los resultados **no serán iguales** a los del artículo (que usa una A100 y 1024 o más entornos).

> **Aviso de honestidad:** este cuaderno se probó en CPU con ajustes diminutos (el código corre de principio a fin).
> No se pudo probar en una GPU T4 real de Colab, así que tiempos, memoria y calidad del aprendizaje son estimaciones hasta que lo ejecutes tú.

## 1. Opciones de checkpoints (Google Drive / retomar)
Las sesiones gratuitas de Colab se cortan. Esta celda, **toda comentada por defecto**, te deja elegir:
1. **No guardar en Drive** (por defecto): no tienes que tocar nada.
2. **Guardar checkpoints en Drive**: descomenta las líneas de la opción 2.
3. **Retomar un entrenamiento**: descomenta además `CHECKPOINT_RETOMAR` con la ruta del `.pt`.

Si Colab te pide reiniciar el entorno más abajo, al volver ejecuta de nuevo esta celda (el reinicio borra las variables).

In [ ]:
# CELL:opciones
# Esta celda decide si se guardan checkpoints en Google Drive o si se retoma un entrenamiento.
# POR DEFECTO (todo comentado) = OPCIÓN 1: nada se guarda en Drive. Los .pt se guardan igual en la carpeta
# local "models" de Colab (hace falta para la inferencia), pero se pierden al cerrar la sesión.
# Para otra opción, quita el "# " de las líneas que correspondan.
CARPETA_GUARDADO = "models"       # opción 1 (por defecto): carpeta local, se borra al cerrar la sesión
CHECKPOINT_RETOMAR = ""           # vacío = entrenar desde cero

# --- OPCIÓN 2: guardar checkpoints en Google Drive (sobreviven si la sesión se corta) ---
# from google.colab import drive
# drive.mount("/content/drive")
# CARPETA_GUARDADO = "/content/drive/MyDrive/fasttd3_models"

# --- OPCIÓN 3: retomar un entrenamiento desde un checkpoint .pt ---
# Descomenta también las líneas de la opción 2 si el .pt está en Drive (si no, súbelo a Colab y pon su ruta aquí).
# Se restauran redes, normalizadores y paso global; el buffer y los optimizadores empiezan de cero (igual que el oficial).
# Usa el mismo num_envs que en el entrenamiento original.
# CHECKPOINT_RETOMAR = "/content/drive/MyDrive/fasttd3_models/T1JoystickFlatTerrain__fasttd3__1_final.pt"


## 2. Instalación
Ejecuta esta celda una vez. Si Colab te pide reiniciar el entorno, hazlo (*Entorno de ejecución → Reiniciar sesión*) y vuelve a ejecutar la celda de opciones (sección 1) y sigue desde las importaciones (sección 3).
Se fija `playground==0.2.0` porque la versión 1.1.0 ya no incluye el robot T1.

In [ ]:
# CELL:install
import os, re, subprocess, sys

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return r.returncode, (r.stdout + r.stderr)

rc, out = sh("nvidia-smi")
m = re.search(r"CUDA Version:\s*(\d+)", out)
cuda_major = int(m.group(1)) if (rc == 0 and m) else None
print("GPU detectada:", out.split("\n")[8].strip() if rc == 0 and len(out.split("\n")) > 8 else "ninguna (se usará CPU: será muy lento)")

extra = None
if cuda_major:
    extra = "cuda13" if cuda_major >= 13 else "cuda12"
pkgs = ["playground==0.2.0", "imageio", "imageio-ffmpeg", "tqdm", "matplotlib"]
if extra:
    pkgs.append(f"jax[{extra}]")
print("Instalando:", pkgs)
rc, out = sh("pip install -q " + " ".join(f'"{p}"' for p in pkgs))
print(out[-1500:] if rc != 0 else "pip: listo")

# Renderizado EGL en Colab (necesario para los videos)
sh("apt-get install -y -qq libegl1 libgl1 > /dev/null 2>&1")
icd = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
if cuda_major and not os.path.exists(icd):
    os.makedirs(os.path.dirname(icd), exist_ok=True)
    with open(icd, "w") as f:
        f.write('{"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}')
print("Si pip actualizó jax, reinicia el entorno de ejecución antes de continuar.")

## 3. Importaciones y comprobaciones

In [ ]:
# CELL:imports
import os, sys, time, math, random, base64, io, dataclasses, json
os.environ["TORCHDYNAMO_INLINE_INBUILT_NN_MODULES"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ.setdefault("MUJOCO_GL", "egl")
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"   # JAX no reserva toda la GPU (la comparte con PyTorch)
os.environ["JAX_DEFAULT_MATMUL_PRECISION"] = "highest"

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.amp import autocast, GradScaler
import jax
import jax.numpy as jnp
import imageio.v2 as imageio
import matplotlib.pyplot as plt
from IPython.display import HTML, display, clear_output
from tqdm.auto import tqdm

torch.set_float32_matmul_precision("high")
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| jax", jax.__version__, "| jax devices:", jax.devices())
print("dispositivo PyTorch:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")
if DEVICE.type == "cpu":
    print("AVISO: no hay GPU. Activa Entorno de ejecución → Cambiar tipo → T4 GPU.")
if DEVICE.type == "cuda" and "gpu" not in {d.platform for d in jax.devices()}:
    print("AVISO: PyTorch ve la GPU pero JAX no; la simulación irá en CPU. Revisa la celda de instalación.")

## 4. Hiperparámetros
Son los valores del repositorio oficial (`hyperparams.py`) para MuJoCo Playground, con los ajustes para T4 aparte en la sección de perfiles (sección 11).

In [ ]:
# CELL:args
@dataclasses.dataclass
class Args:
    # --- hiperparámetros oficiales (BaseArgs + MuJoCoPlaygroundArgs) ---
    env_name: str = "CheetahRun"
    seed: int = 1
    torch_deterministic: bool = True
    num_envs: int = 1024
    num_eval_envs: int = 1024
    total_timesteps: int = 150000
    critic_learning_rate: float = 3e-4
    actor_learning_rate: float = 3e-4
    critic_learning_rate_end: float = 3e-4
    actor_learning_rate_end: float = 3e-4
    buffer_size: int = 1024 * 10        # por entorno
    num_steps: int = 1                  # retorno de n pasos
    gamma: float = 0.97
    tau: float = 0.1
    batch_size: int = 32768
    policy_noise: float = 0.001
    std_min: float = 0.001
    std_max: float = 0.4
    learning_starts: int = 10
    policy_frequency: int = 2
    noise_clip: float = 0.5
    num_updates: int = 2
    init_scale: float = 0.01
    num_atoms: int = 101
    v_min: float = -10.0
    v_max: float = 10.0
    critic_hidden_dim: int = 1024
    actor_hidden_dim: int = 512
    use_cdq: bool = True
    obs_normalization: bool = True
    use_grad_norm_clipping: bool = False
    max_grad_norm: float = 0.0
    amp: bool = True
    disable_bootstrap: bool = False
    weight_decay: float = 0.1
    use_domain_randomization: bool = False
    use_push_randomization: bool = False
    use_tuned_reward: bool = False
    # --- cambios propios del cuaderno ---
    amp_dtype: str = "auto"             # "auto": bf16 si la GPU lo soporta, si no fp16 (T4 -> fp16)
    compile: bool = False               # oficial: True. Sin probar en T4; actívalo bajo tu riesgo
    compile_mode: str = "reduce-overhead"
    mjx_impl: str = "jax"               # "jax" (más compatible) o "warp"
    checkpoint_path: str = ""           # para reanudar (como el oficial: no restaura optimizadores ni buffer)
    eval_interval: int = 1000
    eval_steps: int = 0                 # 0 = episodio completo
    video_interval: int = 1000
    video_steps: int = 250              # pasos de simulación por video
    video_width: int = 320
    video_height: int = 240
    video_in_place: bool = False        # True: reemplaza el video anterior; False: los apila para ver la evolución
    log_interval: int = 100
    save_interval: int = 1000
    save_dir: str = "models"

PRESETS = {  # valores de las clases de hyperparams.py
    "CheetahRun": dict(num_steps=3, v_min=-500.0, v_max=500.0, std_min=0.1, policy_noise=0.1),
    "T1JoystickFlatTerrain": dict(total_timesteps=100000),
    "T1JoystickRoughTerrain": dict(total_timesteps=100000),
    "G1JoystickFlatTerrain": dict(total_timesteps=100000),
    "Go1JoystickFlatTerrain": dict(total_timesteps=50000, std_min=0.2, std_max=0.8, policy_noise=0.2, num_updates=8),
}

def make_args(env_name, **overrides):
    a = Args(env_name=env_name)
    for k, v in {**PRESETS.get(env_name, {}), **overrides}.items():
        if not hasattr(a, k):
            raise ValueError(f"argumento desconocido: {k}")
        setattr(a, k, v)
    return a

## 5. Entornos (MuJoCo Playground + puente JAX ↔ PyTorch)
Las observaciones pasan de JAX a PyTorch sin copiar (dlpack). Al terminar un episodio el entorno se reinicia solo,
así que se guarda la observación *anterior al reinicio* (`raw_obs`) para almacenar la transición correcta, igual que hace el código oficial.

In [ ]:
# CELL:envs
from mujoco_playground import registry
from mujoco_playground import wrapper as pgw
from brax.envs.wrappers import training as brax_training
import mujoco
import torch.utils.dlpack as tpack
from jax.dlpack import from_dlpack as jax_from_dlpack

HUMANOID_TASKS = ["G1JoystickRoughTerrain", "G1JoystickFlatTerrain", "T1JoystickRoughTerrain", "T1JoystickFlatTerrain"]

def _j2t(x):
    return tpack.from_dlpack(x)

def _t2j(x):
    return jax_from_dlpack(x)

def _load_cfg(args, training):
    cfg = registry.get_default_config(args.env_name)
    if "impl" in cfg:
        cfg.impl = args.mjx_impl
    humanoid = args.env_name in HUMANOID_TASKS
    if training and args.use_tuned_reward and humanoid:
        s = cfg.reward_config.scales
        s.energy, s.action_rate, s.torques, s.pose = -5e-5, -1e-1, -1e-3, -1.0
        s.tracking_ang_vel, s.tracking_lin_vel, s.feet_phase = 1.25, 1.25, 1.0
        s.ang_vel_xy, s.orientation = -0.3, -5.0
    if humanoid and not args.use_push_randomization:
        cfg.push_config.enable = False
        cfg.push_config.magnitude_range = [0.0, 0.0]
    return cfg

class RawObsRecorder(pgw.Wrapper):
    # Guarda la observación previa al auto-reinicio en info["raw_obs"].
    def reset(self, rng):
        s = self.env.reset(rng)
        s.info["raw_obs"] = s.obs
        return s
    def step(self, state, action):
        s = self.env.step(state, action)
        s.info["raw_obs"] = s.obs
        return s

class PlaygroundTrainEnv:
    def __init__(self, args):
        cfg = _load_cfg(args, training=True)
        raw_env = registry.load(args.env_name, config=cfg)
        if args.use_domain_randomization:
            raise NotImplementedError("La randomización de dominio no está incluida en este cuaderno.")
        self.num_envs = args.num_envs
        key = jax.random.PRNGKey(args.seed)
        key_reset, _ = jax.random.split(key)
        self.key_reset = jax.random.split(key_reset, args.num_envs)
        env = brax_training.VmapWrapper(raw_env)
        env = brax_training.EpisodeWrapper(env, cfg.episode_length, cfg.get("action_repeat", 1))
        env = RawObsRecorder(env)
        env = pgw.BraxAutoResetWrapper(env, full_reset=False)
        self.env = env
        self.raw_env = raw_env
        obs_shape = raw_env.observation_size
        self.asymmetric_obs = isinstance(obs_shape, dict)
        if self.asymmetric_obs:
            self.num_obs = obs_shape["state"] if isinstance(obs_shape["state"], int) else obs_shape["state"][0]
            p = obs_shape["privileged_state"]
            self.num_privileged_obs = p if isinstance(p, int) else p[0]
        else:
            self.num_obs = obs_shape if isinstance(obs_shape, int) else obs_shape[0]
            self.num_privileged_obs = None
        self.num_actions = raw_env.action_size
        self.max_episode_steps = cfg.episode_length
        self.reset_fn = jax.jit(self.env.reset)
        self.step_fn = jax.jit(self.env.step)
        self.state = None

    def reset_with_critic_obs(self):
        self.state = self.reset_fn(self.key_reset)
        if self.asymmetric_obs:
            return _j2t(self.state.obs["state"]), _j2t(self.state.obs["privileged_state"])
        return _j2t(self.state.obs), None

    def step(self, actions):
        a = _t2j(torch.clip(actions, -1.0, 1.0).contiguous())
        self.state = self.step_fn(self.state, a)
        st = self.state
        reward, done = _j2t(st.reward), _j2t(st.done)
        trunc = _j2t(st.info["truncation"])
        raw = st.info["raw_obs"]
        if self.asymmetric_obs:
            obs, critic = _j2t(st.obs["state"]), _j2t(st.obs["privileged_state"])
            raw_d = {"obs": _j2t(raw["state"]), "critic_obs": _j2t(raw["privileged_state"])}
        else:
            obs, critic = _j2t(st.obs), None
            raw_d = {"obs": _j2t(raw), "critic_obs": None}
        infos = {"time_outs": trunc, "observations": {"critic": critic, "raw": raw_d}}
        return obs, reward, done, infos


class PlaygroundEvalEnv:
    # Entorno sin auto-reinicio para evaluar y renderizar (como PlaygroundEvalEnvWrapper del oficial).
    def __init__(self, args, num_envs, seed=None):
        cfg = _load_cfg(args, training=False)
        self.env = registry.load(args.env_name, config=cfg)
        self.env_name = args.env_name
        self.num_envs = num_envs
        self.max_episode_steps = cfg.episode_length
        self.jit_reset = jax.jit(jax.vmap(self.env.reset))
        self.jit_step = jax.jit(jax.vmap(self.env.step))
        self.asymmetric_obs = isinstance(self.env.observation_size, dict)
        self.key_reset = jax.random.split(jax.random.PRNGKey(args.seed if seed is None else seed), num_envs)
        self.has_command = None

    def reset(self):
        self.state = self.jit_reset(self.key_reset)
        o = self.state.obs["state"] if self.asymmetric_obs else self.state.obs
        return _j2t(o)

    def set_command(self, cmd):
        if "command" in self.state.info:
            self.state.info["command"] = jnp.tile(jnp.asarray([cmd], dtype=jnp.float32), (self.num_envs, 1))

    def step(self, actions):
        self.state = self.jit_step(self.state, _t2j(torch.clip(actions, -1.0, 1.0).contiguous()))
        o = self.state.obs["state"] if self.asymmetric_obs else self.state.obs
        return _j2t(o), _j2t(self.state.reward), _j2t(self.state.done), None

    def render_frames(self, states, width, height):
        opt = mujoco.MjvOption()
        opt.flags[mujoco.mjtVisFlag.mjVIS_TRANSPARENT] = False
        opt.flags[mujoco.mjtVisFlag.mjVIS_PERTFORCE] = False
        opt.flags[mujoco.mjtVisFlag.mjVIS_CONTACTFORCE] = False
        frames = self.env.render(states, camera="track" if "Joystick" in self.env_name else None,
                                 height=height, width=width, scene_option=opt)
        return [np.asarray(f) for f in frames]

## 6. Redes: actor y crítico distribucional (C51)
Copia fiel de `fast_td3.py`: actor MLP 512-256-128 con `tanh`; dos críticos MLP 1024-512-256 que predicen una distribución sobre 101 átomos.
`projection` es la proyección categórica del objetivo.

In [ ]:
# CELL:nets
class DistributionalQNetwork(nn.Module):
    def __init__(self, n_obs, n_act, num_atoms, v_min, v_max, hidden_dim, device=None):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_obs + n_act, hidden_dim, device=device), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2, device=device), nn.ReLU())
        self.fc_head = nn.Sequential(
            nn.Linear(hidden_dim // 2, hidden_dim // 4, device=device), nn.ReLU(),
            nn.Linear(hidden_dim // 4, num_atoms, device=device))
        self.v_min, self.v_max, self.num_atoms = v_min, v_max, num_atoms

    def forward(self, obs, actions):
        return self.fc_head(self.net(torch.cat([obs, actions], 1)))

    def projection(self, obs, actions, rewards, bootstrap, discount, q_support, device):
        delta_z = (self.v_max - self.v_min) / (self.num_atoms - 1)
        batch_size = rewards.shape[0]
        target_z = rewards.unsqueeze(1) + bootstrap.unsqueeze(1) * discount.unsqueeze(1) * q_support
        target_z = target_z.clamp(self.v_min, self.v_max)
        b = (target_z - self.v_min) / delta_z
        l = torch.floor(b).long()
        u = torch.ceil(b).long()
        is_int = l == u
        l_mask = is_int & (l > 0)
        u_mask = is_int & (l == 0)
        l = torch.where(l_mask, l - 1, l)
        u = torch.where(u_mask, u + 1, u)
        next_dist = F.softmax(self.forward(obs, actions), dim=1)
        proj_dist = torch.zeros_like(next_dist)
        offset = (torch.linspace(0, (batch_size - 1) * self.num_atoms, batch_size, device=device)
                  .unsqueeze(1).expand(batch_size, self.num_atoms).long())
        proj_dist.view(-1).index_add_(0, (l + offset).view(-1), (next_dist * (u.float() - b)).view(-1))
        proj_dist.view(-1).index_add_(0, (u + offset).view(-1), (next_dist * (b - l.float())).view(-1))
        return proj_dist


class Critic(nn.Module):
    def __init__(self, n_obs, n_act, num_atoms, v_min, v_max, hidden_dim, device=None):
        super().__init__()
        kw = dict(n_obs=n_obs, n_act=n_act, num_atoms=num_atoms, v_min=v_min, v_max=v_max,
                  hidden_dim=hidden_dim, device=device)
        self.qnet1 = DistributionalQNetwork(**kw)
        self.qnet2 = DistributionalQNetwork(**kw)
        self.register_buffer("q_support", torch.linspace(v_min, v_max, num_atoms, device=device))
        self.device = device

    def forward(self, obs, actions):
        return self.qnet1(obs, actions), self.qnet2(obs, actions)

    def projection(self, obs, actions, rewards, bootstrap, discount):
        d = self.q_support.device
        return (self.qnet1.projection(obs, actions, rewards, bootstrap, discount, self.q_support, d),
                self.qnet2.projection(obs, actions, rewards, bootstrap, discount, self.q_support, d))

    def get_value(self, probs):
        return torch.sum(probs * self.q_support, dim=1)


class Actor(nn.Module):
    def __init__(self, n_obs, n_act, num_envs, init_scale, hidden_dim, std_min=0.05, std_max=0.8, device=None):
        super().__init__()
        self.n_act = n_act
        self.net = nn.Sequential(
            nn.Linear(n_obs, hidden_dim, device=device), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2, device=device), nn.ReLU())
        self.fc_head = nn.Sequential(nn.Linear(hidden_dim // 2, hidden_dim // 4, device=device), nn.ReLU())
        self.fc_mu = nn.Sequential(nn.Linear(hidden_dim // 4, n_act, device=device), nn.Tanh())
        nn.init.normal_(self.fc_mu[0].weight, 0.0, init_scale)
        nn.init.constant_(self.fc_mu[0].bias, 0.0)
        noise_scales = torch.rand(num_envs, 1, device=device) * (std_max - std_min) + std_min
        self.register_buffer("noise_scales", noise_scales)       # un nivel de ruido distinto por entorno
        self.register_buffer("std_min", torch.as_tensor(std_min, device=device))
        self.register_buffer("std_max", torch.as_tensor(std_max, device=device))
        self.n_envs = num_envs
        self.device = device

    def forward(self, obs):
        return self.fc_mu(self.fc_head(self.net(obs)))

    def explore(self, obs, dones=None, deterministic=False):
        if dones is not None and dones.sum() > 0:             # re-sortea el ruido de los entornos que terminaron
            new_scales = torch.rand(self.n_envs, 1, device=obs.device) * (self.std_max - self.std_min) + self.std_min
            dones_view = dones.view(-1, 1) > 0
            self.noise_scales.copy_(torch.where(dones_view, new_scales, self.noise_scales))
        act = self(obs)
        if deterministic:
            return act
        return act + torch.randn_like(act) * self.noise_scales

## 7. Normalización de observaciones y buffer de repetición en GPU
`EmpiricalNormalization` y `SimpleReplayBuffer` siguen a `fast_td3_utils.py` (retorno de n pasos incluido).
Como en el oficial, para entornos asimétricos solo se guarda la parte *privilegiada* de la observación del crítico
(el resto es igual a la observación del actor; se comprobó para el T1: `privilegiada[:85] == estado`).

In [ ]:
# CELL:utils
class EmpiricalNormalization(nn.Module):
    def __init__(self, shape, device, eps=1e-2, until=None):
        super().__init__()
        self.eps, self.until, self.device = eps, until, device
        self.register_buffer("_mean", torch.zeros(shape).unsqueeze(0).to(device))
        self.register_buffer("_var", torch.ones(shape).unsqueeze(0).to(device))
        self.register_buffer("_std", torch.ones(shape).unsqueeze(0).to(device))
        self.register_buffer("count", torch.tensor(0, dtype=torch.long).to(device))

    @property
    def mean(self): return self._mean.squeeze(0).clone()
    @property
    def std(self): return self._std.squeeze(0).clone()

    @torch.no_grad()
    def forward(self, x, center=True, update=True):
        if x.shape[1:] != self._mean.shape[1:]:
            raise ValueError(f"Expected input of shape (*,{self._mean.shape[1:]}), got {x.shape}")
        if self.training and update:
            self.update(x)
        if center:
            return (x - self._mean) / (self._std + self.eps)
        return x / (self._std + self.eps)

    @torch.no_grad()
    def update(self, x):
        if self.until is not None and self.count >= self.until:
            return
        n = x.shape[0]
        batch_mean = torch.mean(x, dim=0, keepdim=True)
        batch_var = torch.var(x, dim=0, keepdim=True, unbiased=False)
        new_count = self.count + n
        delta = batch_mean - self._mean
        self._mean.copy_(self._mean + delta * (n / new_count))
        delta2 = batch_mean - self._mean
        m_a = self._var * self.count
        m_b = batch_var * n
        M2 = m_a + m_b + delta2.pow(2) * (self.count * n / new_count)
        self._var.copy_(M2 / new_count)
        self._std.copy_(self._var.sqrt())
        self.count.copy_(new_count)


class SimpleReplayBuffer(nn.Module):
    def __init__(self, n_env, buffer_size, n_obs, n_act, n_critic_obs, asymmetric_obs=False,
                 playground_mode=False, n_steps=1, gamma=0.99, device=None):
        super().__init__()
        self.n_env, self.buffer_size, self.n_obs, self.n_act = n_env, buffer_size, n_obs, n_act
        self.n_critic_obs, self.asymmetric_obs = n_critic_obs, asymmetric_obs
        self.playground_mode = playground_mode and asymmetric_obs
        if asymmetric_obs and not self.playground_mode:
            raise NotImplementedError("Solo se soporta el modo playground para observaciones asimétricas.")
        self.gamma, self.n_steps, self.device = gamma, n_steps, device
        z = lambda *s, dt=torch.float: torch.zeros(s, device=device, dtype=dt)
        self.observations = z(n_env, buffer_size, n_obs)
        self.actions = z(n_env, buffer_size, n_act)
        self.rewards = z(n_env, buffer_size)
        self.dones = z(n_env, buffer_size, dt=torch.long)
        self.truncations = z(n_env, buffer_size, dt=torch.long)
        self.next_observations = z(n_env, buffer_size, n_obs)
        if asymmetric_obs:
            self.privileged_obs_size = n_critic_obs - n_obs
            self.privileged_observations = z(n_env, buffer_size, self.privileged_obs_size)
            self.next_privileged_observations = z(n_env, buffer_size, self.privileged_obs_size)
        self.ptr = 0

    @torch.no_grad()
    def extend(self, td):
        ptr = self.ptr % self.buffer_size
        self.observations[:, ptr] = td["observations"]
        self.actions[:, ptr] = td["actions"]
        self.rewards[:, ptr] = td["next"]["rewards"]
        self.dones[:, ptr] = td["next"]["dones"]
        self.truncations[:, ptr] = td["next"]["truncations"]
        self.next_observations[:, ptr] = td["next"]["observations"]
        if self.asymmetric_obs:
            self.privileged_observations[:, ptr] = td["critic_observations"][:, self.n_obs:]
            self.next_privileged_observations[:, ptr] = td["next"]["critic_observations"][:, self.n_obs:]
        self.ptr += 1

    @torch.no_grad()
    def sample(self, batch_size):
        B = self.n_env * batch_size
        current_pos = None
        if self.n_steps == 1:
            indices = torch.randint(0, min(self.buffer_size, self.ptr), (self.n_env, batch_size), device=self.device)
            obs_idx = indices.unsqueeze(-1).expand(-1, -1, self.n_obs)
            act_idx = indices.unsqueeze(-1).expand(-1, -1, self.n_act)
            observations = torch.gather(self.observations, 1, obs_idx).reshape(B, self.n_obs)
            next_observations = torch.gather(self.next_observations, 1, obs_idx).reshape(B, self.n_obs)
            actions = torch.gather(self.actions, 1, act_idx).reshape(B, self.n_act)
            rewards = torch.gather(self.rewards, 1, indices).reshape(B)
            dones = torch.gather(self.dones, 1, indices).reshape(B)
            truncations = torch.gather(self.truncations, 1, indices).reshape(B)
            effective_n_steps = torch.ones_like(dones)
            if self.asymmetric_obs:
                pi = indices.unsqueeze(-1).expand(-1, -1, self.privileged_obs_size)
                priv = torch.gather(self.privileged_observations, 1, pi).reshape(B, self.privileged_obs_size)
                next_priv = torch.gather(self.next_privileged_observations, 1, pi).reshape(B, self.privileged_obs_size)
                critic_observations = torch.cat([observations, priv], dim=1)
                next_critic_observations = torch.cat([next_observations, next_priv], dim=1)
        else:
            if self.ptr >= self.buffer_size:
                # Buffer lleno: se marca temporalmente la última posición como truncada para no cruzar episodios
                current_pos = self.ptr % self.buffer_size
                curr_truncations = self.truncations[:, current_pos - 1].clone()
                self.truncations[:, current_pos - 1] = torch.logical_not(self.dones[:, current_pos - 1])
                indices = torch.randint(0, self.buffer_size, (self.n_env, batch_size), device=self.device)
            else:
                max_start_idx = max(1, self.ptr - self.n_steps + 1)
                indices = torch.randint(0, max_start_idx, (self.n_env, batch_size), device=self.device)
            obs_idx = indices.unsqueeze(-1).expand(-1, -1, self.n_obs)
            act_idx = indices.unsqueeze(-1).expand(-1, -1, self.n_act)
            observations = torch.gather(self.observations, 1, obs_idx).reshape(B, self.n_obs)
            actions = torch.gather(self.actions, 1, act_idx).reshape(B, self.n_act)
            if self.asymmetric_obs:
                pi = indices.unsqueeze(-1).expand(-1, -1, self.privileged_obs_size)
                priv = torch.gather(self.privileged_observations, 1, pi).reshape(B, self.privileged_obs_size)
                critic_observations = torch.cat([observations, priv], dim=1)

            seq_offsets = torch.arange(self.n_steps, device=self.device).view(1, 1, -1)
            all_indices = (indices.unsqueeze(-1) + seq_offsets) % self.buffer_size
            ex = lambda t: t.unsqueeze(-1).expand(-1, -1, self.n_steps)
            all_rewards = torch.gather(ex(self.rewards), 1, all_indices)
            all_dones = torch.gather(ex(self.dones), 1, all_indices)
            all_truncations = torch.gather(ex(self.truncations), 1, all_indices)

            all_dones_shifted = torch.cat([torch.zeros_like(all_dones[:, :, :1]), all_dones[:, :, :-1]], dim=2)
            done_masks = torch.cumprod(1.0 - all_dones_shifted, dim=2)
            effective_n_steps = done_masks.sum(2)
            discounts = torch.pow(self.gamma, torch.arange(self.n_steps, device=self.device))
            n_step_rewards = (all_rewards * done_masks * discounts.view(1, 1, -1)).sum(dim=2)

            first_done = torch.argmax((all_dones > 0).float(), dim=2)
            first_trunc = torch.argmax((all_truncations > 0).float(), dim=2)
            no_dones = all_dones.sum(dim=2) == 0
            no_truncs = all_truncations.sum(dim=2) == 0
            first_done = torch.where(no_dones, self.n_steps - 1, first_done)
            first_trunc = torch.where(no_truncs, self.n_steps - 1, first_trunc)
            final_indices = torch.minimum(first_done, first_trunc)
            final_next_obs_indices = torch.gather(all_indices, 2, final_indices.unsqueeze(-1)).squeeze(-1)

            final_next_observations = self.next_observations.gather(
                1, final_next_obs_indices.unsqueeze(-1).expand(-1, -1, self.n_obs))
            final_dones = self.dones.gather(1, final_next_obs_indices)
            final_truncations = self.truncations.gather(1, final_next_obs_indices)
            next_observations = final_next_observations.reshape(B, self.n_obs)
            if self.asymmetric_obs:
                fnp = self.next_privileged_observations.gather(
                    1, final_next_obs_indices.unsqueeze(-1).expand(-1, -1, self.privileged_obs_size))
                next_critic_observations = torch.cat([next_observations, fnp.reshape(B, self.privileged_obs_size)], dim=1)
            rewards = n_step_rewards.reshape(B)
            dones = final_dones.reshape(B)
            truncations = final_truncations.reshape(B)
            effective_n_steps = effective_n_steps.reshape(B)

        out = {"observations": observations, "actions": actions,
               "next": {"rewards": rewards, "dones": dones, "truncations": truncations,
                        "observations": next_observations, "effective_n_steps": effective_n_steps}}
        if self.asymmetric_obs:
            out["critic_observations"] = critic_observations
            out["next"]["critic_observations"] = next_critic_observations
        if self.n_steps > 1 and self.ptr >= self.buffer_size:
            self.truncations[:, current_pos - 1] = curr_truncations
        return out

## 8. Agente FastTD3 (actualizaciones de crítico y actor)
`update_main` es la pérdida del crítico (entropía cruzada contra la distribución proyectada, con mínimo de los dos críticos).
`update_pol` es la actualización retardada del actor (maximiza el valor Q esperado). `soft_update` actualiza el crítico objetivo con `tau`.

In [ ]:
# CELL:agent
def resolve_amp(args):
    enabled = bool(args.amp and DEVICE.type == "cuda")
    if args.amp_dtype == "auto":
        native_bf16 = enabled and torch.cuda.get_device_capability()[0] >= 8   # T4 es 7.5: sin bf16 nativo
        dtype = torch.bfloat16 if native_bf16 else torch.float16
    else:
        dtype = torch.bfloat16 if args.amp_dtype == "bf16" else torch.float16
    return enabled, dtype


class FastTD3Agent:
    def __init__(self, args, n_obs, n_critic_obs, n_act, asymmetric_obs):
        self.args, self.device, self.asymmetric_obs = args, DEVICE, asymmetric_obs
        self.amp_enabled, self.amp_dtype = resolve_amp(args)
        self.scaler = GradScaler(DEVICE.type, enabled=self.amp_enabled and self.amp_dtype == torch.float16)
        if args.obs_normalization:
            self.obs_normalizer = EmpiricalNormalization(shape=n_obs, device=DEVICE)
            self.critic_obs_normalizer = EmpiricalNormalization(shape=n_critic_obs, device=DEVICE)
        else:
            self.obs_normalizer = nn.Identity()
            self.critic_obs_normalizer = nn.Identity()
        self.actor = Actor(n_obs=n_obs, n_act=n_act, num_envs=args.num_envs, device=DEVICE,
                           init_scale=args.init_scale, hidden_dim=args.actor_hidden_dim,
                           std_min=args.std_min, std_max=args.std_max)
        ckw = dict(n_obs=n_critic_obs, n_act=n_act, num_atoms=args.num_atoms, v_min=args.v_min,
                   v_max=args.v_max, hidden_dim=args.critic_hidden_dim, device=DEVICE)
        self.qnet = Critic(**ckw)
        self.qnet_target = Critic(**ckw)
        self.qnet_target.load_state_dict(self.qnet.state_dict())
        self.q_optimizer = optim.AdamW(list(self.qnet.parameters()), lr=args.critic_learning_rate, weight_decay=args.weight_decay)
        self.actor_optimizer = optim.AdamW(list(self.actor.parameters()), lr=args.actor_learning_rate, weight_decay=args.weight_decay)
        self.q_scheduler = optim.lr_scheduler.CosineAnnealingLR(self.q_optimizer, T_max=args.total_timesteps, eta_min=args.critic_learning_rate_end)
        self.actor_scheduler = optim.lr_scheduler.CosineAnnealingLR(self.actor_optimizer, T_max=args.total_timesteps, eta_min=args.actor_learning_rate_end)
        self.update_main_fn, self.update_pol_fn = self._update_main, self._update_pol
        self.normalize_obs = self.obs_normalizer.forward
        self.normalize_critic_obs = self.critic_obs_normalizer.forward
        if args.compile:
            self.update_main_fn = torch.compile(self._update_main, mode=args.compile_mode)
            self.update_pol_fn = torch.compile(self._update_pol, mode=args.compile_mode)
            self.normalize_obs = torch.compile(self.obs_normalizer.forward, mode=None)
            self.normalize_critic_obs = torch.compile(self.critic_obs_normalizer.forward, mode=None)

    def autocast(self):
        return autocast(device_type=self.device.type, dtype=self.amp_dtype, enabled=self.amp_enabled)

    def _update_main(self, data, logs):
        args, qnet, qnet_target, actor = self.args, self.qnet, self.qnet_target, self.actor
        with self.autocast():
            observations = data["observations"]
            next_observations = data["next"]["observations"]
            if self.asymmetric_obs:
                critic_observations = data["critic_observations"]
                next_critic_observations = data["next"]["critic_observations"]
            else:
                critic_observations, next_critic_observations = observations, next_observations
            actions = data["actions"]
            rewards = data["next"]["rewards"]
            dones = data["next"]["dones"].bool()
            truncations = data["next"]["truncations"].bool()
            bootstrap = (~dones).float() if args.disable_bootstrap else (truncations | ~dones).float()

            clipped_noise = torch.randn_like(actions).mul(args.policy_noise).clamp(-args.noise_clip, args.noise_clip)
            next_state_actions = (actor(next_observations) + clipped_noise).clamp(-1.0, 1.0)
            discount = args.gamma ** data["next"]["effective_n_steps"]

            with torch.no_grad():
                q1_proj, q2_proj = qnet_target.projection(next_critic_observations, next_state_actions, rewards, bootstrap, discount)
                q1_val = qnet_target.get_value(q1_proj)
                q2_val = qnet_target.get_value(q2_proj)
                if args.use_cdq:
                    dist = torch.where(q1_val.unsqueeze(1) < q2_val.unsqueeze(1), q1_proj, q2_proj)
                    q1_dist = q2_dist = dist
                else:
                    q1_dist, q2_dist = q1_proj, q2_proj

            qf1, qf2 = qnet(critic_observations, actions)
            qf1_loss = -torch.sum(q1_dist * F.log_softmax(qf1, dim=1), dim=1).mean()
            qf2_loss = -torch.sum(q2_dist * F.log_softmax(qf2, dim=1), dim=1).mean()
            qf_loss = qf1_loss + qf2_loss

        self.q_optimizer.zero_grad(set_to_none=True)
        self.scaler.scale(qf_loss).backward()
        self.scaler.unscale_(self.q_optimizer)
        if args.use_grad_norm_clipping:
            gn = torch.nn.utils.clip_grad_norm_(qnet.parameters(), max_norm=args.max_grad_norm if args.max_grad_norm > 0 else float("inf"))
        else:
            gn = torch.tensor(0.0, device=self.device)
        self.scaler.step(self.q_optimizer)
        self.scaler.update()
        logs["critic_grad_norm"] = gn.detach()
        logs["qf_loss"] = qf_loss.detach()
        logs["qf_max"] = q1_val.max().detach()
        logs["qf_min"] = q1_val.min().detach()
        return logs

    def _update_pol(self, data, logs):
        args, qnet, actor = self.args, self.qnet, self.actor
        with self.autocast():
            critic_observations = data["critic_observations"] if self.asymmetric_obs else data["observations"]
            qf1, qf2 = qnet(critic_observations, actor(data["observations"]))
            qf1_value = qnet.get_value(F.softmax(qf1, dim=1))
            qf2_value = qnet.get_value(F.softmax(qf2, dim=1))
            qf_value = torch.minimum(qf1_value, qf2_value) if args.use_cdq else (qf1_value + qf2_value) / 2.0
            actor_loss = -qf_value.mean()
        self.actor_optimizer.zero_grad(set_to_none=True)
        self.scaler.scale(actor_loss).backward()
        self.scaler.unscale_(self.actor_optimizer)
        if args.use_grad_norm_clipping:
            gn = torch.nn.utils.clip_grad_norm_(actor.parameters(), max_norm=args.max_grad_norm if args.max_grad_norm > 0 else float("inf"))
        else:
            gn = torch.tensor(0.0, device=self.device)
        self.scaler.step(self.actor_optimizer)
        self.scaler.update()
        logs["actor_grad_norm"] = gn.detach()
        logs["actor_loss"] = actor_loss.detach()
        return logs

    @torch.no_grad()
    def soft_update(self):
        tau = self.args.tau
        src = [p.data for p in self.qnet.parameters()]
        tgt = [p.data for p in self.qnet_target.parameters()]
        torch._foreach_mul_(tgt, 1.0 - tau)
        torch._foreach_add_(tgt, src, alpha=tau)

    def prepare_batch(self, data):
        data["observations"] = self.normalize_obs(data["observations"])
        data["next"]["observations"] = self.normalize_obs(data["next"]["observations"])
        if self.asymmetric_obs:
            data["critic_observations"] = self.normalize_critic_obs(data["critic_observations"])
            data["next"]["critic_observations"] = self.normalize_critic_obs(data["next"]["critic_observations"])
        return data


def save_params(global_step, agent, args, save_path):
    os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
    cpu = lambda sd: {k: v.detach().to("cpu") for k, v in sd.items()}
    sd = lambda m: cpu(m.state_dict()) if hasattr(m, "state_dict") else None
    torch.save({"actor_state_dict": sd(agent.actor), "qnet_state_dict": sd(agent.qnet),
                "qnet_target_state_dict": sd(agent.qnet_target),
                "obs_normalizer_state": sd(agent.obs_normalizer),
                "critic_obs_normalizer_state": sd(agent.critic_obs_normalizer),
                "args": dataclasses.asdict(args), "global_step": global_step}, save_path)
    print(f"Checkpoint guardado: {save_path}")

## 9. Videos, evaluación y bucle de entrenamiento
Cada `video_interval` iteraciones se hace un rollout corto con la política **sin ruido** y el video aparece en la salida de la celda.
`evaluate` es la evaluación del oficial (episodio completo con la política determinista).

In [ ]:
# CELL:train
def frames_to_html(frames, fps, caption, width_px=None):
    buf = io.BytesIO()
    tag = None
    try:
        path = "/tmp/_fasttd3_clip.mp4"
        imageio.mimwrite(path, frames, fps=fps, macro_block_size=1)
        b64 = base64.b64encode(open(path, "rb").read()).decode()
        tag = f'<video src="data:video/mp4;base64,{b64}" autoplay loop muted playsinline controls' + (f' width="{width_px}"' if width_px else "") + "></video>"
    except Exception as e:  # sin ffmpeg: GIF
        imageio.mimwrite(buf, frames, format="gif", duration=1000.0 / fps, loop=0)
        b64 = base64.b64encode(buf.getvalue()).decode()
        tag = f'<img src="data:image/gif;base64,{b64}"' + (f' width="{width_px}"' if width_px else "") + ">"
    return f"<div style='margin:6px 0'><b>{caption}</b><br>{tag}</div>"


def show_clip(frames, fps, caption, width_px=None, display_id=None):
    html = HTML(frames_to_html(frames, fps, caption, width_px))
    if display_id is None:
        display(html)
    else:
        try:
            display(html, display_id=display_id)
        except Exception:
            pass
    return html


@torch.no_grad()
def rollout_frames(agent, render_env, args, n_steps, command=(1.0, 0.0, 0.0), every=2):
    obs = render_env.reset()
    render_env.set_command(list(command))
    states = [jax.tree.map(lambda x: x[0], render_env.state)]
    ret = 0.0
    for i in range(n_steps):
        with agent.autocast():
            actions = agent.actor(agent.normalize_obs(obs, update=False) if args.obs_normalization else obs)
        obs, r, done, _ = render_env.step(actions.float())
        render_env.set_command(list(command))
        ret += float(r[0])
        if i % every == 0:
            states.append(jax.tree.map(lambda x: x[0], render_env.state))
        if bool(done.any()):
            break
    frames = render_env.render_frames(states, args.video_width, args.video_height)
    return frames, ret, i + 1


@torch.no_grad()
def evaluate(agent, eval_env, args):
    n = eval_env.num_envs
    returns = torch.zeros(n, device=DEVICE)
    lengths = torch.zeros(n, device=DEVICE)
    done_masks = torch.zeros(n, dtype=torch.bool, device=DEVICE)
    obs = eval_env.reset()
    steps = args.eval_steps if args.eval_steps > 0 else eval_env.max_episode_steps
    for i in range(steps):
        with agent.autocast():
            o = agent.normalize_obs(obs, update=False) if args.obs_normalization else obs
            actions = agent.actor(o)
        next_obs, rewards, dones, _ = eval_env.step(actions.float())
        returns = torch.where(~done_masks, returns + rewards, returns)
        lengths = torch.where(~done_masks, lengths + 1, lengths)
        done_masks = torch.logical_or(done_masks, dones.bool())
        if done_masks.all():
            break
        obs = next_obs
    return returns.mean().item(), lengths.mean().item()


def train(args):
    print(args)
    run_name = f"{args.env_name}__fasttd3__{args.seed}"
    random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)
    torch.backends.cudnn.deterministic = args.torch_deterministic

    envs = PlaygroundTrainEnv(args)
    eval_env = PlaygroundEvalEnv(args, args.num_eval_envs)
    render_env = PlaygroundEvalEnv(args, 1)
    n_act, n_obs = envs.num_actions, envs.num_obs
    n_critic_obs = envs.num_privileged_obs if envs.asymmetric_obs else n_obs

    agent = FastTD3Agent(args, n_obs, n_critic_obs, n_act, envs.asymmetric_obs)
    rb = SimpleReplayBuffer(n_env=args.num_envs, buffer_size=args.buffer_size, n_obs=n_obs, n_act=n_act,
                            n_critic_obs=n_critic_obs, asymmetric_obs=envs.asymmetric_obs,
                            playground_mode=True, n_steps=args.num_steps, gamma=args.gamma, device=DEVICE)
    print(f"obs actor={n_obs} | obs crítico={n_critic_obs} | acciones={n_act} | asimétrico={envs.asymmetric_obs} | "
          f"AMP={agent.amp_enabled} ({agent.amp_dtype})")

    obs, critic_obs = envs.reset_with_critic_obs()
    if critic_obs is not None:
        critic_obs = torch.as_tensor(critic_obs, device=DEVICE, dtype=torch.float)
    global_step = 0
    if args.checkpoint_path:
        ck = torch.load(args.checkpoint_path, map_location=DEVICE, weights_only=False)
        agent.actor.load_state_dict(ck["actor_state_dict"])
        agent.obs_normalizer.load_state_dict(ck["obs_normalizer_state"])
        agent.critic_obs_normalizer.load_state_dict(ck["critic_obs_normalizer_state"])
        agent.qnet.load_state_dict(ck["qnet_state_dict"])
        agent.qnet_target.load_state_dict(ck["qnet_target_state_dict"])
        global_step = ck["global_step"]
        print("Reanudado desde", args.checkpoint_path, "en el paso", global_step)

    history = {"step": [], "eval_return": [], "sps": []}
    dones = None
    logs = {}
    pbar = tqdm(total=args.total_timesteps, initial=global_step)
    t_start, step_start = None, global_step
    video_id = f"vid_{run_name}_{int(time.time())}" if args.video_in_place else None

    def do_video(step, tag="paso"):
        t0 = time.time()
        try:
            frames, ret, n = rollout_frames(agent, render_env, args, args.video_steps)
            fps = max(1, round(1.0 / (2 * 0.02))) if "Joystick" in args.env_name else 30
            last_eval = history["eval_return"][-1] if history["eval_return"] else float("nan")
            show_clip(frames, fps, f"{args.env_name} — {tag} {step} — retorno de este rollout: {ret:.1f} ({n} pasos) — última evaluación: {last_eval:.1f}",
                      width_px=args.video_width, display_id=video_id)
        except Exception as e:
            print("No se pudo generar el video:", repr(e))
        return time.time() - t0

    while global_step < args.total_timesteps:
        if t_start is None and global_step >= args.learning_starts + 3:
            t_start, step_start = time.time(), global_step

        with torch.no_grad(), agent.autocast():
            norm_obs = agent.normalize_obs(obs)
            actions = agent.actor.explore(norm_obs, dones=dones)

        next_obs, rewards, dones, infos = envs.step(actions.float())
        truncations = infos["time_outs"]
        raw = infos["observations"]["raw"]
        true_next_obs = torch.where(dones[:, None] > 0, raw["obs"], next_obs)
        transition = {"observations": obs, "actions": torch.as_tensor(actions, device=DEVICE, dtype=torch.float),
                      "next": {"observations": true_next_obs,
                               "rewards": torch.as_tensor(rewards, device=DEVICE, dtype=torch.float),
                               "truncations": truncations.long(), "dones": dones.long()}}
        if envs.asymmetric_obs:
            next_critic_obs = infos["observations"]["critic"]
            true_next_critic_obs = torch.where(dones[:, None] > 0, raw["critic_obs"], next_critic_obs)
            transition["critic_observations"] = critic_obs
            transition["next"]["critic_observations"] = true_next_critic_obs
        rb.extend(transition)

        obs = next_obs
        if envs.asymmetric_obs:
            critic_obs = next_critic_obs

        if global_step > args.learning_starts:
            for i in range(args.num_updates):
                data = rb.sample(max(1, args.batch_size // args.num_envs))
                data = agent.prepare_batch(data)
                logs = agent.update_main_fn(data, logs)
                if args.num_updates > 1:
                    if i % args.policy_frequency == 1:
                        logs = agent.update_pol_fn(data, logs)
                else:
                    if global_step % args.policy_frequency == 0:
                        logs = agent.update_pol_fn(data, logs)
                agent.soft_update()

            if global_step % args.log_interval == 0 and t_start is not None and "actor_loss" in logs:
                sps = (global_step - step_start) / (time.time() - t_start)
                pbar.set_description(f"{sps:5.1f} it/s | qf_loss {logs['qf_loss'].item():.3f} | actor_loss {logs['actor_loss'].item():.2f} | rew {rewards.mean().item():.3f}")
                history["sps"].append(sps)

            if args.eval_interval > 0 and global_step % args.eval_interval == 0:
                er, el = evaluate(agent, eval_env, args)
                history["step"].append(global_step); history["eval_return"].append(er)
                print(f"[paso {global_step}] evaluación: retorno medio {er:.2f} | largo medio {el:.0f}")
                if t_start is not None:  # el tiempo de evaluación y video no cuenta para la velocidad
                    pass
            if args.video_interval > 0 and global_step % args.video_interval == 0:
                dt = do_video(global_step)
                if t_start is not None:
                    t_start += dt
            if args.save_interval > 0 and global_step > 0 and global_step % args.save_interval == 0:
                save_params(global_step, agent, args, f"{args.save_dir}/{run_name}_{global_step}.pt")

        global_step += 1
        agent.actor_scheduler.step()
        agent.q_scheduler.step()
        pbar.update(1)
    pbar.close()

    er, el = evaluate(agent, eval_env, args)
    history["step"].append(global_step); history["eval_return"].append(er)
    print(f"[final, paso {global_step}] evaluación: retorno medio {er:.2f} | largo medio {el:.0f}")
    do_video(global_step, tag="final, paso")
    final_path = f"{args.save_dir}/{run_name}_final.pt"
    save_params(global_step, agent, args, final_path)
    if len(history["step"]) > 1:
        plt.figure(figsize=(5, 3)); plt.plot(history["step"], history["eval_return"], marker="o")
        plt.xlabel("iteración"); plt.ylabel("retorno de evaluación"); plt.title(args.env_name); plt.grid(alpha=.3); plt.show()
    return {"agent": agent, "history": history, "checkpoint": final_path, "args": args}

## 10. Benchmark: ¿qué tan rápida es tu GPU?
Mide cuánto tarda un paso del simulador y una actualización de las redes, y estima cuántas iteraciones caben en el tiempo que quieras.
**Ejecútalo antes de entrenar** y ajusta los perfiles de la celda siguiente con lo que veas.
También calcula cuánta memoria ocupará el buffer (es lo que más pesa: crece con `num_envs × buffer_size`).

In [ ]:
# CELL:bench
def buffer_gb(num_envs, buffer_size, n_obs, n_act, n_critic_obs):
    priv = max(0, n_critic_obs - n_obs)
    floats = 2 * n_obs + n_act + 1 + 2 * priv
    return num_envs * buffer_size * (floats * 4 + 2 * 8) / 1e9


def _sync(envs=None):
    # JAX es asíncrono: hay que esperar a JAX y a PyTorch para medir bien
    if envs is not None and envs.state is not None:
        jax.block_until_ready(envs.state.obs)
        jax.block_until_ready(envs.state.reward)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()


def benchmark(env_name, num_envs=512, batch_size=8192, iters=20, minutes=(10, 30, 60), **kw):
    args = make_args(env_name, num_envs=num_envs, num_eval_envs=8, batch_size=batch_size, **kw)
    print(f"== {env_name} | {num_envs} entornos | lote {batch_size} ==")
    t = time.time()
    envs = PlaygroundTrainEnv(args)
    obs, critic_obs = envs.reset_with_critic_obs()
    _sync(envs)
    print(f"carga + compilación de reset: {time.time() - t:.1f} s")
    n_obs, n_act = envs.num_obs, envs.num_actions
    n_critic = envs.num_privileged_obs if envs.asymmetric_obs else n_obs
    t = time.time()
    a = torch.zeros(num_envs, n_act, device=DEVICE)
    envs.step(a); _sync(envs)
    envs.step(a); _sync(envs)   # el simulador se recompila una segunda vez: se excluye de la medición
    print(f"compilación de los dos primeros pasos: {time.time() - t:.1f} s")
    t = time.time()
    for _ in range(iters):
        envs.step(torch.rand(num_envs, n_act, device=DEVICE) * 2 - 1)
    _sync(envs)
    t_env = (time.time() - t) / iters
    print(f"simulador: {t_env * 1000:.1f} ms por paso ({num_envs / t_env:,.0f} pasos de entorno/s)")

    agent = FastTD3Agent(args, n_obs, n_critic, n_act, envs.asymmetric_obs)
    B = batch_size
    mk = lambda n: torch.randn(B, n, device=DEVICE)
    def fake():
        d = {"observations": mk(n_obs), "actions": torch.rand(B, n_act, device=DEVICE) * 2 - 1,
             "next": {"observations": mk(n_obs), "rewards": torch.randn(B, device=DEVICE),
                      "dones": torch.zeros(B, dtype=torch.long, device=DEVICE),
                      "truncations": torch.zeros(B, dtype=torch.long, device=DEVICE),
                      "effective_n_steps": torch.ones(B, dtype=torch.long, device=DEVICE)}}
        if envs.asymmetric_obs:
            d["critic_observations"] = mk(n_critic); d["next"]["critic_observations"] = mk(n_critic)
        return d
    data, logs = fake(), {}
    for _ in range(2):  # calentamiento
        logs = agent.update_main_fn(data, logs); logs = agent.update_pol_fn(data, logs); agent.soft_update()
    _sync()
    t = time.time()
    for _ in range(iters):
        logs = agent.update_main_fn(data, logs); logs = agent.update_pol_fn(data, logs); agent.soft_update()
    _sync()
    t_upd = (time.time() - t) / iters
    print(f"actualización (crítico + actor + suave): {t_upd * 1000:.1f} ms")
    t_iter = t_env + args.num_updates * t_upd + 0.002
    print(f"-> iteración completa estimada: {t_iter * 1000:.0f} ms  (~{1 / t_iter:.1f} it/s) con num_updates={args.num_updates}")
    for m in minutes:
        print(f"   en {m:>3} min caben ~{int(m * 60 / t_iter):,} iteraciones (sin contar evaluación ni videos)")
    for bs in (1024, 2048, 4096, 10240):
        print(f"   buffer con {bs} pasos por entorno: {buffer_gb(num_envs, bs, n_obs, n_act, n_critic):.2f} GB")
    if DEVICE.type == "cuda":
        print(f"memoria GPU usada por PyTorch ahora: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB (sin buffer)")
    return t_iter

In [ ]:
# CELL:bench_run_t1
bench_t1 = benchmark("T1JoystickFlatTerrain", num_envs=512, batch_size=8192)

## 11. Perfiles para T4
Ajusta estos números con lo que midió el benchmark. Regla práctica: `total_timesteps ≈ minutos × 60 / segundos_por_iteración`.
Las cifras iniciales son **puntos de partida razonables, no resultados verificados en una T4**.
Referencia oficial (A100): 1024 entornos, lote 32768, buffer 10240 por entorno, ~100 mil iteraciones para T1.

- `video_interval`: cada cuántas iteraciones ves un video (cada video añade unos segundos de renderizado). Aquí vale 1/4 del total, así ves 4 videos: tres durante el entrenamiento y uno final.
- Si te quedas sin memoria: baja `buffer_size` o `num_envs`.

In [ ]:
# CELL:config
TOTAL_T1 = 10000
PERFIL_T1 = dict(num_envs=512, num_eval_envs=64, batch_size=8192, buffer_size=2048,
                 total_timesteps=TOTAL_T1, eval_interval=1000, video_interval=TOTAL_T1 // 4,
                 save_interval=1000, video_steps=250)
PERFIL_T1["save_dir"] = CARPETA_GUARDADO
if CHECKPOINT_RETOMAR:
    PERFIL_T1["checkpoint_path"] = CHECKPOINT_RETOMAR


## 12. Entrenar: humanoide Booster T1 (`T1JoystickFlatTerrain`)
El robot debe seguir una orden de velocidad (adelante/lateral/giro). En los videos se le ordena avanzar a 1 m/s.
Aprender a caminar bien lleva bastante más que CheetahRun; en pocos miles de iteraciones es normal ver solo un comportamiento inestable o que "arrastra" los pies.
Cada `video_interval` iteraciones (1/4 del total) aparece un video. Conviene haber validado antes la instalación con el cuaderno de CheetahRun.

In [ ]:
# CELL:run_t1
args_t1 = make_args("T1JoystickFlatTerrain", **PERFIL_T1)
out_t1 = train(args_t1)

## 13. Inferencia: cargar un checkpoint y ver al robot
Carga un `.pt` guardado (por ejemplo el `_final.pt`) y genera un video más largo con la política determinista, sin ruido de exploración.
Puedes cambiar la orden de velocidad `command = (vx, vy, giro)`.

In [ ]:
# CELL:infer
def run_inference(ckpt_path, steps=500, command=(1.0, 0.0, 0.0), width=480, height=360):
    ck = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    a = Args(**{k: v for k, v in ck["args"].items() if k in Args.__dataclass_fields__})
    a.video_width, a.video_height = width, height
    env = PlaygroundEvalEnv(a, 1)
    n_obs = env.env.observation_size["state"] if env.asymmetric_obs else env.env.observation_size
    n_obs = n_obs if isinstance(n_obs, int) else n_obs[0]
    n_act = env.env.action_size
    agent = FastTD3Agent.__new__(FastTD3Agent)          # solo hace falta el actor y el normalizador
    agent.args, agent.device, agent.asymmetric_obs = a, DEVICE, env.asymmetric_obs
    agent.amp_enabled, agent.amp_dtype = False, torch.float32
    agent.actor = Actor(n_obs=n_obs, n_act=n_act, num_envs=a.num_envs, device=DEVICE, init_scale=a.init_scale,
                        hidden_dim=a.actor_hidden_dim, std_min=a.std_min, std_max=a.std_max)
    agent.actor.load_state_dict(ck["actor_state_dict"])
    agent.obs_normalizer = EmpiricalNormalization(shape=n_obs, device=DEVICE)
    agent.obs_normalizer.load_state_dict(ck["obs_normalizer_state"])
    agent.obs_normalizer.eval()
    agent.normalize_obs = agent.obs_normalizer.forward
    agent.autocast = lambda: autocast(device_type=DEVICE.type, enabled=False)
    frames, ret, n = rollout_frames(agent, env, a, steps, command=command)
    fps = 25 if "Joystick" in a.env_name else 30
    show_clip(frames, fps, f"Inferencia — {a.env_name} — paso {ck['global_step']} — retorno {ret:.1f} en {n} pasos", width_px=width)
    return frames

# Ejemplo (descomenta):
# frames = run_inference(out_t1["checkpoint"], steps=500, command=(1.0, 0.0, 0.0))